# Public English → Icelandic adapters

Upload this notebook to Colab and choose a GPU. This notebook performs inference only. Public model downloads need no HF login. Base-model/data terms still apply. The reduced-precision default is for ordinary inference and is not a reproduction of the FP32 benchmark. No GitHub checkout is needed.


In [ ]:
%pip -q install transformers==5.22.1 peft==0.21.2 accelerate==1.15.0 safetensors==0.8.0 sentencepiece==0.2.2


In [ ]:
MODELS = {'qwen25-3b-sft-r0': {'repo': 'huaq99/mtrl-qwen25-3b-en-is-sft-r0', 'revision': 'ae660dd4ea272fff7caf840d2c9ccc76401e6780', 'base': 'Qwen/Qwen2.5-3B-Instruct', 'base_revision': 'aa8e72537993ba99e69dfaafa59ed015b17504d1', 'architecture': 'qwen2'}, 'qwen25-3b-dapo-shortmid': {'repo': 'huaq99/mtrl-qwen25-3b-en-is-dapo-shortmid-v1', 'revision': '1ec9cfaa813582bda509676497da2ffb2e69e6bc', 'base': 'Qwen/Qwen2.5-3B-Instruct', 'base_revision': 'aa8e72537993ba99e69dfaafa59ed015b17504d1', 'architecture': 'qwen2'}, 'qwen35-08b-sft-r0': {'repo': 'huaq99/mtrl-qwen35-08b-en-is-sft-r0', 'revision': 'c95800cbefea66cbe449a482c7515a74a6984564', 'base': 'Qwen/Qwen3.5-0.8B', 'base_revision': '2fc06364715b967f1860aea9cf38778875588b17', 'architecture': 'qwen3_5'}}
MODEL_ID = "qwen25-3b-dapo-shortmid"  # or either SFT baseline
PRECISION = "float16"  # lower-memory ordinary inference; FP32 for the recorded 3B path


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
assert torch.cuda.is_available(), "Select a GPU runtime before loading weights"
spec = MODELS[MODEL_ID]
dtype = getattr(torch, PRECISION)
tokenizer = AutoTokenizer.from_pretrained(spec['base'], revision=spec['base_revision'])
if spec['architecture'] == 'qwen3_5':
    from transformers import Qwen3_5ForConditionalGeneration
    constructor = Qwen3_5ForConditionalGeneration
else:
    constructor = AutoModelForCausalLM
base = constructor.from_pretrained(spec['base'], revision=spec['base_revision'], dtype=dtype, attn_implementation='sdpa')
model = PeftModel.from_pretrained(base, spec['repo'], revision=spec['revision'], is_trainable=False).to(dtype=dtype, device='cuda').eval()


In [ ]:
INSTRUCTION = 'Translate the following English text into Icelandic. Return only the Icelandic translation.'
source = "Put the book on the table."
messages = [{'role':'user', 'content': INSTRUCTION+'\n\n'+source}]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
inputs = tokenizer(prompt, add_special_tokens=False, return_tensors='pt').to('cuda')
with torch.inference_mode():
    output = model.generate(**inputs, do_sample=False, num_beams=1, max_new_tokens=512, pad_token_id=tokenizer.pad_token_id or tokenizer.eos_token_id)
print(tokenizer.decode(output[0, inputs.input_ids.shape[1]:], skip_special_tokens=True))
